<a href="https://colab.research.google.com/github/cizred/AI/blob/main/%E8%BB%8A%E7%89%8C%E8%BE%A8%E8%AD%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

車牌辨識

In [1]:
!pip -q install ultralytics

# 1) 下載你放在 GitHub Release 的 dataset zip
# 下面 URL 請改成你的 Release 連結（tag + 檔名要對）
!wget -O dataset.zip \
https://github.com/cizred/taiwan-license-plate-number-recognition/releases/download/dataset/taiwan-license-plate-char-recognition-research.v1-----.yolov8.zip


# 2) 解壓
!unzip -q dataset.zip -d dataset


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.0 MB/s eta 0:00:00
--2025-12-15 03:15:08--  https://github.com/cizred/taiwan-license-plate-number-recognition/releases/download/dataset/taiwan-license-plate-char-recognition-research.v1-----.yolov8.zip
Resolving github.com (github.com)... 140.82.112.3
Connecting to github.com (github.com)|140.82.112.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/1116526948/15134ef6-5253-4f70-a915-d60f491330fe?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-12-15T03%3A56%3A04Z&rscd=attachment%3B+filename%3Dtaiwan-license-plate-char-recognition-research.v1-----.yolov8.zip&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-12-15T02%3A55%3A09Z&ske=2025-12-15T03%3A56%3A04Z&sks=b&skv=2018-11-09&sig=2UlrKJXVHmSrsDOZ%2BttjuDba%2FdGFpJteIbGNfXDRbNo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhb

RuntimeError: Dataset 'dataset/data.yaml' error ❌ Dataset 'dataset/data.yaml' images not found, missing path '/content/dataset/valid/images'
Note dataset download directory is '/content/datasets'. You can update this in '/root/.config/Ultralytics/settings.json'

In [2]:
import os, random, shutil, glob, yaml

base = "dataset"
train_img = os.path.join(base, "train/images")
train_lbl = os.path.join(base, "train/labels")
val_img   = os.path.join(base, "val/images")
val_lbl   = os.path.join(base, "val/labels")

os.makedirs(val_img, exist_ok=True)
os.makedirs(val_lbl, exist_ok=True)

# 取所有訓練影像
imgs = sorted(glob.glob(os.path.join(train_img, "*.*")))
assert len(imgs) > 0, "train/images 沒有影像"

# 抽 10% 當 val
random.seed(42)
val_count = max(1, int(len(imgs) * 0.1))
val_imgs = set(random.sample(imgs, val_count))

def label_path(img_path):
    stem = os.path.splitext(os.path.basename(img_path))[0]
    return os.path.join(train_lbl, stem + ".txt")

moved = 0
missing_labels = 0

for img in val_imgs:
    lbl = label_path(img)
    shutil.move(img, os.path.join(val_img, os.path.basename(img)))
    if os.path.exists(lbl):
        shutil.move(lbl, os.path.join(val_lbl, os.path.basename(lbl)))
    else:
        missing_labels += 1
    moved += 1

print(f"Moved {moved} images to val. Missing labels: {missing_labels}")

# 讀原 data.yaml
yaml_in = os.path.join(base, "data.yaml")
with open(yaml_in, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

# 寫新的 data_fixed.yaml（用 path + 相對路徑）
data["path"] = os.path.abspath(base)
data["train"] = "train/images"
data["val"] = "val/images"

yaml_out = os.path.join(base, "data_fixed.yaml")
with open(yaml_out, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print("Wrote:", yaml_out)
print(open(yaml_out, "r", encoding="utf-8").read())


Moved 78 images to val. Missing labels: 0
Wrote: dataset/data_fixed.yaml
train: train/images
val: val/images
test: ../test/images
nc: 36
names:
- '-'
- '0'
- '1'
- '2'
- '3'
- '4'
- '5'
- '6'
- '7'
- '8'
- '9'
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
roboflow:
  workspace: new-workspace-ceodd
  project: taiwan-license-plate-char-recognition-research-3fnbd
  version: 1
  license: CC BY 4.0
  url: https://universe.roboflow.com/new-workspace-ceodd/taiwan-license-plate-char-recognition-research-3fnbd/dataset/1
path: /content/dataset



In [ ]:
!pip -q install ultralytics
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
model.train(data="dataset/data_fixed.yaml", epochs=100, imgsz=640, batch=16)


Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data_fixed.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=train2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0.0, plots=True, pose=12.0